# Phase 10 — Feature Experiments

**Goal:** test feature groups independently and determine whether temporal, lag, and rolling information improves next-hour PM2.5 prediction.

Each experiment has its **own complete code block**. No shared experiment function is used.

All comparisons use the same chronological walk-forward validation folds.

## 1. Load engineered data and folds

In [9]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))

from src.project_config import *
from src.modeling_utils import *

import json
import numpy as np
import pandas as pd

if not FEATURE_DATA.exists():
    raise FileNotFoundError("Run 05_feature_engineering.ipynb first.")
features = pd.read_csv(FEATURE_DATA, parse_dates=["datetime", "target_datetime"])

manifest_path = SPLIT_DIR / "walk_forward_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError("Run 06_data_split.ipynb first.")
manifest = pd.read_csv(manifest_path)

if PHASE9_SELECTION.exists():
    EXPERIMENT_MODEL = json.loads(PHASE9_SELECTION.read_text(encoding="utf-8"))["selected_model"]
else:
    raise FileNotFoundError("Run 09_model_comparison.ipynb after all four models have been evaluated.")

print("Experiment model:", EXPERIMENT_MODEL)
print("Engineered rows:", f"{len(features):,}")
display(manifest[["fold", "train_rows", "validation_rows"]])

Experiment model: Random Forest
Engineered rows: 420,768


,fold,train_rows,validation_rows
0,fold_1,82553,87927
1,fold_2,170480,87088
2,fold_3,257568,86826


## 2. Define feature groups

In [10]:
BASE_NUMERIC = [
    "year", "month", "day", "hour",
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "WSPM",
]
BASE_CATEGORICAL = ["wd", "station"]
TEMPORAL_FEATURES = ["day_of_week", "is_weekend", "season", "hour_sin", "hour_cos"]
LAG_FEATURES = [f"PM2.5_lag_{x}h" for x in LAGS]
ROLLING_FEATURES = [
    *[f"PM2.5_roll_mean_{x}h" for x in ROLL_MEANS],
    *[f"PM2.5_roll_std_{x}h" for x in ROLL_STDS],
]

FEATURE_GROUPS = {
    "A_basic": BASE_NUMERIC + BASE_CATEGORICAL,
    "B_basic_plus_temporal": BASE_NUMERIC + BASE_CATEGORICAL + TEMPORAL_FEATURES,
    "C_basic_plus_temporal_plus_lag": BASE_NUMERIC + BASE_CATEGORICAL + TEMPORAL_FEATURES + LAG_FEATURES,
    "D_all_including_rolling": BASE_NUMERIC + BASE_CATEGORICAL + TEMPORAL_FEATURES + LAG_FEATURES + ROLLING_FEATURES,
}

for group_name, group_features in FEATURE_GROUPS.items():
    print(group_name, "→", len(group_features), "raw feature columns")

A_basic → 17 raw feature columns
B_basic_plus_temporal → 22 raw feature columns
C_basic_plus_temporal_plus_lag → 28 raw feature columns
D_all_including_rolling → 33 raw feature columns


## 3. Experiment A — Basic features

In [11]:
# Independent Experiment A block
RUN_EXPERIMENT_BASIC = True
results_A = None
EXPERIMENT_FEATURES = FEATURE_GROUPS["A_basic"]

if RUN_EXPERIMENT_BASIC:
    fold_results = []
    for _, item in manifest.iterrows():
        train = pd.read_csv(SPLIT_DIR / item["train_file"])
        validation = pd.read_csv(SPLIT_DIR / item["validation_file"])

        missing_train = [c for c in EXPERIMENT_FEATURES if c not in train.columns]
        missing_validation = [c for c in EXPERIMENT_FEATURES if c not in validation.columns]
        if missing_train or missing_validation:
            raise ValueError(f"A_basic missing columns. Train={missing_train}, Validation={missing_validation}")

        columns, numeric, categorical = model_input_columns(train, TARGET, feature_columns=EXPERIMENT_FEATURES)
        X_train, y_train = train[columns], train[TARGET]
        X_validation, y_validation = validation[columns], validation[TARGET]
        pipeline = make_model_pipeline(EXPERIMENT_MODEL, numeric, categorical)
        pipeline.fit(X_train, y_train)

        train_prediction = pipeline.predict(X_train)
        validation_prediction = pipeline.predict(X_validation)
        train_metrics = regression_metrics(y_train, train_prediction)
        validation_metrics = regression_metrics(y_validation, validation_prediction)

        fold_results.append({
            "experiment": "A_basic", "fold": item["fold"], "feature_count": len(columns),
            "train_RMSE": train_metrics["RMSE"], "validation_RMSE": validation_metrics["RMSE"],
            "train_MAE": train_metrics["MAE"], "validation_MAE": validation_metrics["MAE"],
            "train_R2": train_metrics["R2"], "validation_R2": validation_metrics["R2"],
        })
    results_A = pd.DataFrame(fold_results)
    display(results_A.round(4))
else:
    print("Experiment A skipped.")

,experiment,fold,feature_count,train_RMSE,validation_RMSE,train_MAE,validation_MAE,train_R2,validation_R2
0,A_basic,fold_1,17,11.7567,22.5744,6.6085,11.6446,0.9743,0.9179
1,A_basic,fold_2,17,12.0131,26.3226,6.8280,11.8944,0.9752,0.9025
2,A_basic,fold_3,17,12.2662,20.4382,6.9853,10.0256,0.9759,0.9275


## 4. Experiment B — Basic + temporal/cyclical features

In [17]:
# Independent Experiment B block
RUN_EXPERIMENT_TEMPORAL = True
results_B = None
EXPERIMENT_FEATURES = FEATURE_GROUPS["B_basic_plus_temporal"]

if RUN_EXPERIMENT_TEMPORAL:
    fold_results = []
    for _, item in manifest.iterrows():
        train = pd.read_csv(SPLIT_DIR / item["train_file"])
        validation = pd.read_csv(SPLIT_DIR / item["validation_file"])
        missing_train = [c for c in EXPERIMENT_FEATURES if c not in train.columns]
        missing_validation = [c for c in EXPERIMENT_FEATURES if c not in validation.columns]
        if missing_train or missing_validation:
            raise ValueError(f"B_basic_plus_temporal missing columns. Train={missing_train}, Validation={missing_validation}")

        columns, numeric, categorical = model_input_columns(train, TARGET, feature_columns=EXPERIMENT_FEATURES)
        X_train, y_train = train[columns], train[TARGET]
        X_validation, y_validation = validation[columns], validation[TARGET]
        pipeline = make_model_pipeline(EXPERIMENT_MODEL, numeric, categorical)
        pipeline.fit(X_train, y_train)

        train_prediction = pipeline.predict(X_train)
        validation_prediction = pipeline.predict(X_validation)
        train_metrics = regression_metrics(y_train, train_prediction)
        validation_metrics = regression_metrics(y_validation, validation_prediction)

        fold_results.append({
            "experiment": "B_basic_plus_temporal", "fold": item["fold"], "feature_count": len(columns),
            "train_RMSE": train_metrics["RMSE"], "validation_RMSE": validation_metrics["RMSE"],
            "train_MAE": train_metrics["MAE"], "validation_MAE": validation_metrics["MAE"],
            "train_R2": train_metrics["R2"], "validation_R2": validation_metrics["R2"],
        })
    results_B = pd.DataFrame(fold_results)
    display(results_B.round(4))
else:
    print("Experiment B skipped.")

,experiment,fold,feature_count,train_RMSE,validation_RMSE,train_MAE,validation_MAE,train_R2,validation_R2
0,B_basic_plus_temporal,fold_1,22,11.5567,22.7757,6.4706,11.6634,0.9752,0.9164
1,B_basic_plus_temporal,fold_2,22,11.8629,27.4775,6.7373,12.1131,0.9758,0.8938
2,B_basic_plus_temporal,fold_3,22,12.1098,20.3174,6.9065,9.9751,0.9765,0.9283


## 5. Experiment C — Basic + temporal + lag features

In [19]:
# Independent Experiment C block
RUN_EXPERIMENT_LAG = True
results_C = None
EXPERIMENT_FEATURES = FEATURE_GROUPS["C_basic_plus_temporal_plus_lag"]

if RUN_EXPERIMENT_LAG:
    fold_results = []
    for _, item in manifest.iterrows():
        train = pd.read_csv(SPLIT_DIR / item["train_file"])
        validation = pd.read_csv(SPLIT_DIR / item["validation_file"])
        missing_train = [c for c in EXPERIMENT_FEATURES if c not in train.columns]
        missing_validation = [c for c in EXPERIMENT_FEATURES if c not in validation.columns]
        if missing_train or missing_validation:
            raise ValueError(f"C_basic_plus_temporal_plus_lag missing columns. Train={missing_train}, Validation={missing_validation}")

        columns, numeric, categorical = model_input_columns(train, TARGET, feature_columns=EXPERIMENT_FEATURES)
        X_train, y_train = train[columns], train[TARGET]
        X_validation, y_validation = validation[columns], validation[TARGET]
        pipeline = make_model_pipeline(EXPERIMENT_MODEL, numeric, categorical)
        pipeline.fit(X_train, y_train)

        train_prediction = pipeline.predict(X_train)
        validation_prediction = pipeline.predict(X_validation)
        train_metrics = regression_metrics(y_train, train_prediction)
        validation_metrics = regression_metrics(y_validation, validation_prediction)

        fold_results.append({
            "experiment": "C_basic_plus_temporal_plus_lag", "fold": item["fold"], "feature_count": len(columns),
            "train_RMSE": train_metrics["RMSE"], "validation_RMSE": validation_metrics["RMSE"],
            "train_MAE": train_metrics["MAE"], "validation_MAE": validation_metrics["MAE"],
            "train_R2": train_metrics["R2"], "validation_R2": validation_metrics["R2"],
        })
    results_C = pd.DataFrame(fold_results)
    display(results_C.round(4))
else:
    print("Experiment C skipped.")

,experiment,fold,feature_count,train_RMSE,validation_RMSE,train_MAE,validation_MAE,train_R2,validation_R2
0,C_basic_plus_temporal_plus_lag,fold_1,28,11.2574,21.9675,6.3243,11.1981,0.9765,0.9222
1,C_basic_plus_temporal_plus_lag,fold_2,28,11.4765,25.9744,6.5725,11.4245,0.9773,0.9051
2,C_basic_plus_temporal_plus_lag,fold_3,28,11.7791,19.5527,6.7864,9.5198,0.9778,0.9336


## 6. Experiment D — Basic + temporal + lag + rolling features

In [20]:
# Independent Experiment D block
RUN_EXPERIMENT_ROLLING = True
results_D = None
EXPERIMENT_FEATURES = FEATURE_GROUPS["D_all_including_rolling"]

if RUN_EXPERIMENT_ROLLING:
    fold_results = []
    for _, item in manifest.iterrows():
        train = pd.read_csv(SPLIT_DIR / item["train_file"])
        validation = pd.read_csv(SPLIT_DIR / item["validation_file"])
        missing_train = [c for c in EXPERIMENT_FEATURES if c not in train.columns]
        missing_validation = [c for c in EXPERIMENT_FEATURES if c not in validation.columns]
        if missing_train or missing_validation:
            raise ValueError(f"D_all_including_rolling missing columns. Train={missing_train}, Validation={missing_validation}")

        columns, numeric, categorical = model_input_columns(train, TARGET, feature_columns=EXPERIMENT_FEATURES)
        X_train, y_train = train[columns], train[TARGET]
        X_validation, y_validation = validation[columns], validation[TARGET]
        pipeline = make_model_pipeline(EXPERIMENT_MODEL, numeric, categorical)
        pipeline.fit(X_train, y_train)

        train_prediction = pipeline.predict(X_train)
        validation_prediction = pipeline.predict(X_validation)
        train_metrics = regression_metrics(y_train, train_prediction)
        validation_metrics = regression_metrics(y_validation, validation_prediction)

        fold_results.append({
            "experiment": "D_all_including_rolling", "fold": item["fold"], "feature_count": len(columns),
            "train_RMSE": train_metrics["RMSE"], "validation_RMSE": validation_metrics["RMSE"],
            "train_MAE": train_metrics["MAE"], "validation_MAE": validation_metrics["MAE"],
            "train_R2": train_metrics["R2"], "validation_R2": validation_metrics["R2"],
        })
    results_D = pd.DataFrame(fold_results)
    display(results_D.round(4))
else:
    print("Experiment D skipped.")

,experiment,fold,feature_count,train_RMSE,validation_RMSE,train_MAE,validation_MAE,train_R2,validation_R2
0,D_all_including_rolling,fold_1,33,11.1669,21.8413,6.2887,11.1635,0.9768,0.9231
1,D_all_including_rolling,fold_2,33,11.4207,25.2119,6.5580,11.2881,0.9775,0.9106
2,D_all_including_rolling,fold_3,33,11.7660,19.2640,6.7963,9.5057,0.9778,0.9356


## 7. Compare feature experiments

In [21]:
experiment_results = [r for r in [results_A, results_B, results_C, results_D] if r is not None and not r.empty]

if len(experiment_results) == 4:
    combined = pd.concat(experiment_results, ignore_index=True)
    feature_comparison = (
        combined.groupby("experiment", as_index=False)
        .agg(
            feature_count=("feature_count", "first"),
            mean_validation_RMSE=("validation_RMSE", "mean"),
            validation_RMSE_std=("validation_RMSE", lambda s: s.std(ddof=0)),
            mean_validation_MAE=("validation_MAE", "mean"),
            mean_validation_R2=("validation_R2", "mean"),
            mean_train_RMSE=("train_RMSE", "mean"),
        )
    )
    feature_comparison["train_to_validation_RMSE_gap"] = (
        feature_comparison["mean_validation_RMSE"] - feature_comparison["mean_train_RMSE"]
    )
    display(feature_comparison.round(4))

    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    feature_comparison.to_csv(PHASE10_RESULTS, index=False)
    print("Created:", PHASE10_RESULTS)
else:
    feature_comparison = None
    print("Run all four feature experiment blocks before making the Phase 10 feature decision.")

,experiment,feature_count,mean_validation_RMSE,validation_RMSE_std,mean_validation_MAE,mean_validation_R2,mean_train_RMSE,train_to_validation_RMSE_gap
0,A_basic,17,23.1117,2.4321,11.1882,0.9160,12.0120,11.0998
1,B_basic_plus_temporal,22,23.5235,2.9705,11.2506,0.9128,11.8431,11.6804
2,C_basic_plus_temporal_plus_lag,28,22.4982,2.6484,10.7141,0.9203,11.5044,10.9938
3,D_all_including_rolling,33,22.1057,2.4354,10.6524,0.9231,11.4512,10.6545


Created: /Users/ruchi/Downloads/phase12_fixed/data/results/phase10_feature_experiments.csv


## 8. Select the feature group using validation evidence only

In [22]:
if feature_comparison is not None and not feature_comparison.empty:
    selected_feature_group = str(
        feature_comparison.sort_values(
            ["mean_validation_RMSE", "mean_validation_MAE", "validation_RMSE_std"]
        ).iloc[0]["experiment"]
    )

    selection = {
        "selected_feature_group": selected_feature_group,
        "selection_metric": "mean_validation_RMSE",
        "tie_breakers": ["mean_validation_MAE", "validation_RMSE_std"],
        "final_test_used": False,
        "feature_columns": FEATURE_GROUPS[selected_feature_group],
    }
    PHASE10_SELECTION.write_text(json.dumps(selection, indent=2), encoding="utf-8")
    print("Selected feature group using validation only:", selected_feature_group)
    print("Created:", PHASE10_SELECTION)
else:
    print("Feature selection record was not created because all four experiments were not completed.")

Selected feature group using validation only: D_all_including_rolling
Created: /Users/ruchi/Downloads/phase12_fixed/data/results/phase10_selected_feature_group.json


## 9. Decision rule

A feature group is supported when it improves validation performance consistently enough to justify its added complexity. The protected 2017 holdout is not used here.